<a href="https://colab.research.google.com/github/Lingesh3012/Html-profolio/blob/main/AI_Meal_Planner_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🍱 AI Meal Planner Agent

A complete Google Colab project that calculates nutrition targets and generates a 7-day Indian meal plan.

## Install packages

In [1]:
!pip -q install gradio pandas


## Imports

In [2]:
import random
import math
import pandas as pd
import gradio as gr


## Food database

In [3]:
# =========================
FOODS = [
    # name, category, diet, serving_g, calories, protein_g
    ("Oats", "breakfast", "veg", 100, 389, 16.9),
    ("Idli", "breakfast", "veg", 150, 174, 6),
    ("Dosa", "breakfast", "veg", 150, 250, 6),
    ("Vegetable Upma", "breakfast", "veg", 200, 250, 7),
    ("Poha", "breakfast", "veg", 200, 250, 6),
    ("Sambar Idli", "breakfast", "veg", 250, 280, 10),
    ("Curd Rice", "lunch", "veg", 300, 350, 10),
    ("Rice + Dal", "lunch", "veg", 350, 450, 16),
    ("Rice + Rajma", "lunch", "veg", 350, 480, 18),
    ("Rice + Chana", "lunch", "veg", 350, 470, 18),
    ("Vegetable Rice", "lunch", "veg", 300, 380, 9),
    ("Chapati + Dal", "lunch", "veg", 300, 420, 17),
    ("Chicken Rice", "lunch", "nonveg", 400, 600, 42),
    ("Chicken + Rice + Vegetables", "lunch", "nonveg", 450, 650, 48),
    ("Chicken Breast + Rice", "lunch", "nonveg", 400, 620, 55),
    ("Fish + Rice", "lunch", "nonveg", 400, 550, 42),
    ("Chicken Sandwich", "snack", "nonveg", 250, 400, 30),
    ("Peanut Banana Shake", "snack", "veg", 350, 450, 15),
    ("Sattu Drink", "snack", "veg", 300, 250, 12),
    ("Roasted Chana", "snack", "veg", 60, 220, 12),
    ("Peanuts", "snack", "veg", 50, 285, 13),
    ("Banana", "snack", "veg", 120, 105, 1.3),
    ("Apple", "snack", "veg", 180, 95, 0.5),
    ("Curd", "snack", "veg", 200, 120, 7),
    ("Milk", "snack", "veg", 250, 150, 8),
    ("Paneer Rice Bowl", "dinner", "veg", 400, 550, 27),
    ("Dal Rice + Vegetables", "dinner", "veg", 400, 500, 18),
    ("Chapati + Paneer", "dinner", "veg", 300, 480, 25),
    ("Vegetable Dosa", "dinner", "veg", 250, 350, 10),
    ("Chicken + Chapati", "dinner", "nonveg", 350, 550, 45),
    ("Chicken + Vegetables", "dinner", "nonveg", 350, 500, 50),
    ("Fish + Vegetables", "dinner", "nonveg", 350, 450, 40),
]

FOOD_DF = pd.DataFrame(
    FOODS,
    columns=["food", "meal_type", "diet", "serving_g", "calories", "protein_g"]
)


## Calculations

In [4]:
# =========================
def calculate_targets(age, sex, height_cm, weight_kg, activity, goal):
    """
    Mifflin-St Jeor BMR + activity multiplier.
    This is a general educational estimate.
    """

    age = float(age)
    height_cm = float(height_cm)
    weight_kg = float(weight_kg)

    if sex.lower() == "male":
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age + 5
    else:
        bmr = 10 * weight_kg + 6.25 * height_cm - 5 * age - 161

    activity_factors = {
        "Sedentary": 1.20,
        "Light": 1.375,
        "Moderate": 1.55,
        "Very Active": 1.725,
        "Athlete": 1.90
    }

    tdee = bmr * activity_factors.get(activity, 1.55)

    # Conservative educational adjustments
    if goal == "Lose Weight":
        calories = tdee - 300
    elif goal == "Gain Muscle":
        calories = tdee + 250
    else:
        calories = tdee

    calories = max(1200, round(calories))

    # General protein target range used for the project
    if goal == "Gain Muscle":
        protein = round(weight_kg * 1.6)
    elif goal == "Lose Weight":
        protein = round(weight_kg * 1.6)
    else:
        protein = round(weight_kg * 1.2)

    return round(bmr), round(tdee), calories, protein


## Meal generator

In [5]:
# =========================
def get_candidates(meal_type, diet):
    df = FOOD_DF[
        (FOOD_DF["meal_type"] == meal_type) &
        ((FOOD_DF["diet"] == "veg") if diet == "Vegetarian"
         else (FOOD_DF["diet"].isin(["veg", "nonveg"])))
    ]
    return df.to_dict("records")


def choose_meal(meal_type, diet, remaining_calories, remaining_protein):
    candidates = get_candidates(meal_type, diet)

    if not candidates:
        return None

    # Score foods according to remaining calorie/protein targets.
    scored = []
    for item in candidates:
        cal_diff = abs(item["calories"] - remaining_calories * 0.25)
        protein_diff = abs(item["protein_g"] - remaining_protein * 0.25)
        score = cal_diff + protein_diff * 8
        scored.append((score, item))

    scored.sort(key=lambda x: x[0])

    # Randomize among the best few for variety.
    top = scored[:min(5, len(scored))]
    return random.choice(top)[1]


def generate_day(diet, calorie_target, protein_target, seed):
    random.seed(seed)

    meal_types = [
        ("Breakfast", "breakfast"),
        ("Morning Snack", "snack"),
        ("Lunch", "lunch"),
        ("Evening Snack", "snack"),
        ("Dinner", "dinner")
    ]

    meals = []
    total_calories = 0
    total_protein = 0

    for display_name, meal_type in meal_types:
        remaining_cal = max(calorie_target - total_calories, 200)
        remaining_protein = max(protein_target - total_protein, 5)

        meal = choose_meal(
            meal_type,
            diet,
            remaining_cal,
            remaining_protein
        )

        if meal:
            meals.append({
                "Meal": display_name,
                "Food": meal["food"],
                "Serving": f'{meal["serving_g"]} g',
                "Calories": int(meal["calories"]),
                "Protein": round(meal["protein_g"], 1)
            })
            total_calories += meal["calories"]
            total_protein += meal["protein_g"]

    # Add a simple fruit if the generated day is too low in calories.
    if total_calories < calorie_target * 0.80:
        fruit = random.choice(
            FOOD_DF[FOOD_DF["food"].isin(["Banana", "Apple"])].to_dict("records")
        )
        meals.append({
            "Meal": "Extra",
            "Food": fruit["food"],
            "Serving": f'{fruit["serving_g"]} g',
            "Calories": int(fruit["calories"]),
            "Protein": round(fruit["protein_g"], 1)
        })
        total_calories += fruit["calories"]
        total_protein += fruit["protein_g"]

    return meals, round(total_calories), round(total_protein, 1)


## 7-day plan

In [6]:
# =========================
def generate_week(diet, calorie_target, protein_target):
    week = []

    for day in range(1, 8):
        meals, cal, protein = generate_day(
            diet,
            calorie_target,
            protein_target,
            seed=day * 1234
        )

        day_df = pd.DataFrame(meals)
        day_df.insert(0, "Day", f"Day {day}")

        week.append(day_df)

    return pd.concat(week, ignore_index=True)


## Gradio app function

In [7]:
# =========================
def create_plan(age, sex, height, weight, activity, goal, diet):
    try:
        bmr, tdee, calories, protein = calculate_targets(
            age, sex, height, weight, activity, goal
        )

        plan = generate_week(
            diet,
            calories,
            protein
        )

        summary = f"""
## AI Meal Planner Result

**Estimated BMR:** {bmr} kcal/day
**Estimated TDEE:** {tdee} kcal/day
**Daily calorie target:** {calories} kcal/day
**Daily protein target:** {protein} g/day

### Goal
{goal}

### Diet
{diet}

> These are general estimates for an educational project. Actual nutrition needs can vary.
"""

        return summary, plan

    except Exception as e:
        return f"Error: {str(e)}", pd.DataFrame()


## Gradio interface

In [8]:
# =========================
with gr.Blocks(title="AI Meal Planner Agent") as demo:

    gr.Markdown(
        """
# 🍱 AI Meal Planner Agent

Enter your details and generate a **7-day personalized meal plan**.

The agent estimates your calorie and protein requirements and selects meals
from an Indian-food database.
"""
    )

    with gr.Row():
        with gr.Column():
            age = gr.Number(
                label="Age",
                value=18,
                minimum=13,
                maximum=100
            )

            sex = gr.Dropdown(
                ["Male", "Female"],
                label="Sex",
                value="Male"
            )

            height = gr.Number(
                label="Height (cm)",
                value=176
            )

            weight = gr.Number(
                label="Weight (kg)",
                value=62
            )

        with gr.Column():
            activity = gr.Dropdown(
                [
                    "Sedentary",
                    "Light",
                    "Moderate",
                    "Very Active",
                    "Athlete"
                ],
                label="Activity Level",
                value="Moderate"
            )

            goal = gr.Dropdown(
                [
                    "Maintain Weight",
                    "Lose Weight",
                    "Gain Muscle"
                ],
                label="Goal",
                value="Gain Muscle"
            )

            diet = gr.Dropdown(
                ["Vegetarian", "Non-Vegetarian"],
                label="Diet Preference",
                value="Non-Vegetarian"
            )

    generate_button = gr.Button(
        "🚀 Generate 7-Day Meal Plan",
        variant="primary"
    )

    summary = gr.Markdown()

    plan_table = gr.Dataframe(
        headers=["Day", "Meal", "Food", "Serving", "Calories", "Protein"],
        datatype=["str", "str", "str", "str", "number", "number"],
        label="7-Day Meal Plan",
        interactive=False
    )

    generate_button.click(
        fn=create_plan,
        inputs=[
            age,
            sex,
            height,
            weight,
            activity,
            goal,
            diet
        ],
        outputs=[
            summary,
            plan_table
        ]
    )

    gr.Markdown(
        """
### Project Architecture

**User Input → BMR/TDEE Calculator → Nutrition Target → Meal Selection Agent → 7-Day Plan → UI**

### Future upgrades

- Add Gemini/OpenAI API for natural-language meal recommendations
- Add allergies and food exclusions
- Add Indian regional foods
- Add budget-based planning
- Add grocery-list generation
- Add calorie/protein charts
- Add meal-swapping
- Save plans to CSV/PDF
- Add chatbot interface
"""
    )


## Launch

In [11]:
demo.launch(share=True)


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://635b6496bbdb7c88dd.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
